In [0]:
# --- Setup for the Silver notebook ---
# `spark` is pre-defined in every Databricks notebook; we only need the functions module.
from pyspark.sql import functions as F
print("ready")

In [0]:
# --- Build the games DataFrame from bronze_scoreboard (nothing is saved yet) ---
from pyspark.sql import functions as F

# 1. The schema: only the fields we want. Everything else in the JSON is ignored.
#    Struct = JSON object, ARRAY = JSON list.
scoreboard_schema = """
events ARRAY<STRUCT<
    id: STRING,
    date: STRING,
    name: STRING,
    status: STRUCT<type: STRUCT<name: STRING>>,
    competitions: ARRAY<STRUCT<
        neutralSite: BOOLEAN,
        competitors: ARRAY<STRUCT<
            homeAway: STRING,
            score: STRING,
            team: STRUCT<id: STRING, abbreviation: STRING, displayName: STRING>
        >>
    >>
>>
"""

# 2. Parse the raw JSON text into a struct column named "j" (runs in parallel)
parsed = spark.table("workspace.nba.bronze_scoreboard").select(
    "source_file",
    F.from_json("raw_json", scoreboard_schema).alias("j"),
)

# 3. explode: turn the array of games in each file into one row per game
events = parsed.select("source_file", F.explode("j.events").alias("e"))

# 4. Pull out the home and away team structs. filter() with a lambda scans the
#    competitors array for the matching team, and [0] takes the first match.
games = events.select(
    "source_file",
    "e",
    F.expr("filter(e.competitions[0].competitors, c -> c.homeAway = 'home')[0]").alias("home"),
    F.expr("filter(e.competitions[0].competitors, c -> c.homeAway = 'away')[0]").alias("away"),
)

# 5. Final column selection: flat names, proper types
silver_games_df = games.select(
    F.col("e.id").alias("game_id"),
    # The date we requested from the API, taken from the Bronze file name
    F.to_date(F.regexp_extract("source_file", r"game_date=(\d{4}-\d{2}-\d{2})", 1)).alias("game_date"),
    # Tip-off time, parsed from text like "2026-10-20T19:00Z"
    F.to_timestamp("e.date", "yyyy-MM-dd'T'HH:mm'Z'").alias("tipoff_utc"),
    F.col("e.name").alias("matchup"),
    F.col("e.status.type.name").alias("status"),
    F.col("e.competitions")[0]["neutralSite"].alias("neutral_site"),
    F.col("home.team.id").alias("home_team_id"),
    F.col("home.team.abbreviation").alias("home_team_abbr"),
    F.col("home.team.displayName").alias("home_team_name"),
    F.col("home.score").cast("int").alias("home_score"),   # string -> integer
    F.col("away.team.id").alias("away_team_id"),
    F.col("away.team.abbreviation").alias("away_team_abbr"),
    F.col("away.team.displayName").alias("away_team_name"),
    F.col("away.score").cast("int").alias("away_score"),
    "source_file",
)

# 6. Inspect: schema first (cheap), then actions that trigger the real work
silver_games_df.printSchema()
print(silver_games_df.count(), "rows")
print(silver_games_df.select("game_id").distinct().count(), "distinct game_ids")
silver_games_df.drop("source_file").show(5, truncate=40)

In [0]:
# --- Data quality checks on silver_games_df (nothing is saved) ---
from pyspark.sql import functions as F

# Helper: counts the rows that BREAK a rule (so 0 means the check passes)
def bad(condition):
    return F.sum(F.when(condition, 1).otherwise(0))

# 1. All row-level checks in ONE aggregation (a single pass over the data)
results = silver_games_df.agg(
    F.count("*").alias("total_rows"),
    # Missing identifiers
    bad(F.col("game_id").isNull() | F.col("home_team_id").isNull() | F.col("away_team_id").isNull()).alias("null_ids"),
    # Missing scores or tip-off time (null comparisons aren't caught by the rules below)
    bad(F.col("home_score").isNull() | F.col("away_score").isNull()).alias("null_scores"),
    bad(F.col("tipoff_utc").isNull()).alias("null_tipoff"),
    # NBA games always have points scored by both teams
    bad((F.col("home_score") <= 0) | (F.col("away_score") <= 0)).alias("non_positive_scores"),
    # NBA games can't end in a tie
    bad(F.col("home_score") == F.col("away_score")).alias("ties"),
    # A team can't play itself
    bad(F.col("home_team_id") == F.col("away_team_id")).alias("same_team"),
    # We only fetched final games
    bad(F.col("status") != "STATUS_FINAL").alias("not_final"),
    # Dates should fall inside the range we backfilled
    bad((F.col("game_date") < F.lit("2026-10-20").cast("date")) |
        (F.col("game_date") > F.lit("2026-11-15").cast("date"))).alias("date_out_of_range"),
    # The UTC tip-off date should be the game date or the day after (late US games)
    bad(~F.datediff(F.to_date("tipoff_utc"), F.col("game_date")).isin(0, 1)).alias("tipoff_date_mismatch"),
).first().asDict()

# 2. Cross-row check: no team should play twice on the same date.
#    Stack home and away teams into one list, then count games per team per date.
team_games = (
    silver_games_df.select("game_date", F.col("home_team_id").alias("team_id"))
    .unionByName(silver_games_df.select("game_date", F.col("away_team_id").alias("team_id")))
)
results["team_twice_same_day"] = (
    team_games.groupBy("game_date", "team_id").count().where("count > 1").count()
)

# 3. Report: PASS when the number of offending rows is 0
total = results.pop("total_rows")
print(f"Checked {total} rows\n")
for name, n_bad in results.items():
    print(f"{'PASS' if n_bad == 0 else 'FAIL'}  {name}: {n_bad}")

In [0]:
# --- Write silver_games as a Delta table ---
# Relies on: silver_games_df

SILVER_GAMES = "workspace.nba.silver_games"   # catalog.schema.table

# The ACTION: Spark now runs the whole plan (read Bronze -> parse -> flatten -> cast)
# and stores the result. overwrite makes the cell safe to re-run.
(
    silver_games_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_GAMES)
)

# Add a description that shows up in the Catalog UI
spark.sql(f"COMMENT ON TABLE {SILVER_GAMES} IS 'One row per final NBA game: date, teams, and scores. Parsed and typed from bronze_scoreboard.'")

# Verify 1: read the saved table back and count its rows
print(spark.table(SILVER_GAMES).count(), "rows in", SILVER_GAMES)

# Verify 2: the transaction log entry for this write, with its metrics
display(
    spark.sql(f"DESCRIBE HISTORY {SILVER_GAMES}")
    .select("version", "operation", "operationMetrics")
)